## Optimize Prompts in Databricks Prompt Registry

### Installing Utilities and Libraries

In [ ]:
%pip install --upgrade "mlflow[databricks]>=3.1.0" databricks-sdk==0.77.0 openai dspy

### Restart the Python Environment

In [ ]:
dbutils.library.restartPython()

### Setup your Environment

In [ ]:
import mlflow
import uuid

mlflow.set_tracking_uri("databricks")

mlflow.set_experiment(
    "/Shared/prompt-optimization"
)

# Get current catalog
catalog_name = spark.sql(
    "SELECT current_catalog()"
).collect()[0][0]

schema_name = "default"

suffix = uuid.uuid4().hex[:6]

PROMPT_NAME = (
    f"{catalog_name}.{schema_name}."
    f"classification_prompt"
)


print("Prompt:", PROMPT_NAME)

### Register Initial Prompt

In [ ]:
prompt = mlflow.genai.register_prompt(
    name=PROMPT_NAME,

    template="""
Classify the following IT support ticket.

Ticket:
{{ticket}}
""",

    commit_message="v1: Basic ticket classification prompt"
)


# set a production alias
mlflow.genai.set_prompt_alias(
    name=PROMPT_NAME,
    alias="development",
    version=1
)

print(f"Prompt: {prompt.name}")
print(f"Version: {prompt.version}")

### Create the LLM Model Client

In [ ]:
from databricks.sdk import WorkspaceClient
import openai

# create the LLM client
llm_client = WorkspaceClient().serving_endpoints.get_open_ai_client()

# Define the model name
model_name = "databricks-claude-sonnet-4-5"

### Define your Prediction Function

In [ ]:
def predict_fn(query):

        prompt = mlflow.genai.load_prompt(
            name_or_uri=(
                f"prompts:/{PROMPT_NAME}@development"
            )
        )

        formatted_prompt = prompt.format(
            ticket=query
        )

        response = llm_client.chat.completions.create(
            model=model_name,
            messages=[
                {
                    "role": "user",
                    "content": formatted_prompt
                }
            ],
            temperature=0.1
        )

        return response.choices[0].message.content

### Test Your Prediction Function

In [ ]:
ticket_query = "My second monitor occasionally flickers, but I can continue working normally"

output = predict_fn(query = ticket_query)


print(output)

### Optimize against Data

In [ ]:
# Training data with inputs, expected outputs, and expectations
dataset = [
    {
        "inputs": {
            "query": (
                "I changed my password this morning and "
                "now I cannot log into my corporate account."
            )
        },
        "outputs": {
            "response": "Category: Authentication\nPriority: High"
        },
        "expectations": {
            "expected_facts": [
                "Category must be 'Authentication'",
                "Priority must be 'High'"
            ]
        }
    },

    {
        "inputs": {
            "query": (
                "The office Wi-Fi is unavailable for everyone "
                "on the third floor and nobody can access "
                "internal applications."
            )
        },
        "outputs": {
            "response": "Category: Network\nPriority: Critical"
        },
        "expectations": {
            "expected_facts": [
                "Category must be 'Network'",
                "Priority must be 'Critical'"
            ]
        }
    },

    {
        "inputs": {
            "query": (
                "Microsoft Excel crashes whenever I try to "
                "open one particular spreadsheet. Other "
                "spreadsheets work normally."
            )
        },
        "outputs": {
            "response": "Category: Software\nPriority: Medium"
        },
        "expectations": {
            "expected_facts": [
                "Category must be 'Software'",
                "Priority must be 'Medium'"
            ]
        }
    },

    {
        "inputs": {
            "query": (
                "I received an email asking me to enter my "
                "company password on an unfamiliar website."
            )
        },
        "outputs": {
            "response": "Category: Security\nPriority: High"
        },
        "expectations": {
            "expected_facts": [
                "Category must be 'Security'",
                "Priority must be 'High'"
            ]
        }
    },

    {
        "inputs": {
            "query": (
                "My second monitor occasionally flickers, "
                "but I can continue working normally."
            )
        },
        "outputs": {
            "response": "Category: Hardware\nPriority: Low"
        },
        "expectations": {
            "expected_facts": [
                "Category must be 'Hardware'",
                "Priority must be 'Low'"
            ]
        }
    }
]

In [ ]:
from mlflow.genai.optimize import GepaPromptOptimizer
from mlflow.genai.scorers import Correctness

# Optimize the prompt
result = mlflow.genai.optimize_prompts(
    predict_fn = predict_fn,
    train_data = dataset,
    prompt_uris = [prompt.uri],
    optimizer = GepaPromptOptimizer(reflection_model = "databricks:/databricks-claude-sonnet-4-5"),
    scorers = [Correctness(model="databricks:/databricks-claude-sonnet-4-5")]
)

# Use the Optimized Prompt
optimized_prompt = result.optimized_prompts[0]
print(f"Optimized Template: {optimized_prompt.template}")

### Use the Optimized Prompt

In [ ]:
VERSION_NUMBER = "LATEST_PROMPT_VERSION_NUMBER"

def predict_fn(query):

        prompt = mlflow.genai.load_prompt(
            name_or_uri=(
                f"prompts:/{PROMPT_NAME}/{VERSION_NUMBER}"
            )
        )

        formatted_prompt = prompt.format(
            ticket=query
        )

        response = llm_client.chat.completions.create(
            model=model_name,
            messages=[
                {
                    "role": "user",
                    "content": formatted_prompt
                }
            ],
            temperature=0.1
        )

        return response.choices[0].message.content

In [ ]:
ticket_query = "My second monitor occasionally flickers, but I can continue working normally"

output = predict_fn(query = ticket_query)


print(output)